In [2]:
# Data manipulation
import pandas as pd
import numpy as np

# SQL / SQLite
import sqlite3

# Visualization
import matplotlib.pyplot as plt
import seaborn as sns

# Statistics
from scipy import stats

# Machine Learning
import sklearn
from sklearn import metrics
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor

# Excel / files
import openpyxl
import pyarrow as pa
import pyarrow.parquet as pq

# System / environment
import sys
import os

print("Environment ready")
print("Python:", sys.version)
print("Pandas:", pd.__version__)
print("NumPy:", np.__version__)
print("SQLite:", sqlite3.sqlite_version)
print("Scikit-learn:", sklearn.__version__)

Environment ready
Python: 3.11.17 | packaged by Anaconda, Inc. | (main, Oct  1 2026, 20:16:54) [MSC v.1942 64 bit (AMD64)]
Pandas: 3.0.6
NumPy: 2.4.6
SQLite: 3.53.4
Scikit-learn: 1.9.1


In [6]:
df_calendar = pd.read_excel("datasets/calendar_master.xlsx")
df_product = pd.read_excel("datasets/product_master.xlsx")
df_market = pd.read_excel("datasets/market_master.xlsx")
df_sales = pd.read_excel("datasets/daily_sales_accounts_2024_2026.xlsx")
df_rates = pd.read_excel("datasets/fx_rate_master_daily.xlsx")

conn = sqlite3.connect(":memory:")

df_calendar.to_sql("dim_calendar", conn, index=False, if_exists="replace")
df_product.to_sql("dim_product", conn, index=False, if_exists="replace")
df_market.to_sql("dim_market", conn, index=False, if_exists="replace")
df_sales.to_sql("fact_sales", conn, index=False, if_exists="replace")
df_rates.to_sql("fact_rates", conn, index=False, if_exists="replace")


5220

In [13]:
query = """
SELECT
    *
FROM fact_sales
LIMIT 5;
"""

pd.read_sql_query(query, conn)

,date,product_code,market_code,market_country,currency,account,value,source_system
0,2024-09-30 00:00:00,PRD_004,MKT_DE,Germany,EUR,sales,198216.34,synthetic_source
1,2024-09-30 00:00:00,PRD_004,MKT_DE,Germany,EUR,accruals,-8764.39,synthetic_source
2,2024-09-30 00:00:00,PRD_004,MKT_DE,Germany,EUR,units,19.00,synthetic_source
3,2024-09-30 00:00:00,PRD_004,MKT_ES,Spain,EUR,sales,49959.26,synthetic_source
4,2024-09-30 00:00:00,PRD_004,MKT_ES,Spain,EUR,accruals,-9346.22,synthetic_source


In [24]:
query = """
WITH sales_metrics AS (
    SELECT
        date AS sale_date,
        product_code,
        market_code,
        market_country,
        currency,
        CASE WHEN account = "sales" THEN value ELSE 0 END AS sales,
        CASE WHEN account = "accruals" THEN value ELSE 0 END AS accruals,
        CASE WHEN account = "units" THEN value ELSE 0 END AS units
    FROM fact_sales
),sales_summary AS (
SELECT
    sale_date,
    product_code,
    market_code,
    market_country,
    currency,
    SUM(sales) AS sales,
    SUM(accruals) AS accruals,
    CAST(SUM(units) AS INTEGER) AS units
FROM sales_metrics
GROUP BY sale_date, product_code, market_code, market_country, currency
),sales_with_rates AS (
    SELECT
        ss.sale_date,
        ss.product_code,
        ss.market_code,
        ss.market_country,
        ss.currency,
        ss.sales,
        ss.accruals,
        ss.units,
        fr.fx_rate_to_usd AS fx_rate,
        (ss.sales+ss.accruals) * fr.fx_rate_to_usd AS real_sales_usd
    FROM sales_summary ss
    LEFT JOIN fact_rates fr
        ON ss.sale_date = fr.rate_date AND ss.currency = fr.currency_code
)  
SELECT * FROM sales_with_rates
"""

pd.read_sql_query(query, conn)

,sale_date,product_code,market_code,market_country,currency,sales,accruals,units,fx_rate,real_sales_usd
0,2024-09-30 00:00:00,PRD_004,MKT_DE,Germany,EUR,198216.34,-8764.39,19,1.075605,203775.455207
1,2024-09-30 00:00:00,PRD_004,MKT_ES,Spain,EUR,49959.26,-9346.22,22,1.075605,43683.586859
2,2024-09-30 00:00:00,PRD_010,MKT_DE,Germany,EUR,14276.72,-827.54,2,1.075605,14466.004581
3,2024-09-30 00:00:00,PRD_010,MKT_MX,Mexico,MXN,96248.65,-6130.45,12,0.055638,5013.962167
4,2024-09-30 00:00:00,PRD_011,MKT_BR,Brazil,BRL,15036.76,-356.30,5,0.180429,2648.779102
...,...,...,...,...,...,...,...,...,...,...
29995,2026-09-29 00:00:00,PRD_183,MKT_US,United States,USD,41491.84,-8078.68,20,1.000000,33413.160000
29996,2026-09-29 00:00:00,PRD_190,MKT_AU,Australia,AUD,64598.54,-3891.50,7,0.676930,41094.404446
29997,2026-09-29 00:00:00,PRD_192,MKT_CO,Colombia,COP,125674.61,-21976.96,16,0.000244,25.342669
29998,2026-09-29 00:00:00,PRD_195,MKT_CL,Chile,CLP,136538.93,-17906.58,14,0.001031,122.332493
